In [1]:
import os
import pandas as pd
import numpy as np
import networkx as nx
from tqdm.auto import tqdm

# ==========================================
# SETTINGS
# ==========================================
GO_OBO_FILE = "/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo"
OUTPUT_TSV = "submission.tsv"

SCORE_CUTOFF = 0.001

# ==========================================
# SMART TSV FINDER (ABSOLUTE SAFE)
# ==========================================
def find_submission_tsv():
    base_dir = "/kaggle/input"
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.endswith(".tsv"):
                path = os.path.join(root, f)
                print(f"[Auto] Found TSV file: {path}")
                return path
    raise FileNotFoundError("No .tsv file found in Kaggle input directory")

# ==========================================
# GO DAG CONSTRUCTION
# ==========================================
def load_go_dag(obo_file):
    print("[Step 1] Loading GO ontology...")
    
    dag = nx.DiGraph()
    current_term = None

    if not os.path.isfile(obo_file):
        raise FileNotFoundError(f"OBO file missing: {obo_file}")

    with open(obo_file, "r") as fh:
        for raw in fh:
            line = raw.strip()

            if line == "[Term]":
                current_term = None
            elif line.startswith("id: "):
                current_term = line.split("id: ")[1]
                dag.add_node(current_term)
            elif current_term and line.startswith("is_a: "):
                dag.add_edge(current_term, line.split()[1])
            elif current_term and line.startswith("relationship: part_of "):
                dag.add_edge(current_term, line.split()[2])

    if not nx.is_directed_acyclic_graph(dag):
        for cycle in nx.simple_cycles(dag):
            dag.remove_edge(cycle[0], cycle[1])

    topo_terms = list(nx.topological_sort(dag))
    parent_map = {t: list(dag.successors(t)) for t in dag.nodes}

    return topo_terms, parent_map

# ==========================================
# SCORE PROPAGATION
# ==========================================
def enforce_hierarchy(df, topo_terms, parent_map):
    print("[Step 3] Propagating scores...")
    
    results = []
    for protein_id, block in tqdm(df.groupby("protein_id")):
        score_map = dict(zip(block.go_term, block.score))

        for term in topo_terms:
            if term not in score_map:
                continue
            val = score_map[term]
            for parent in parent_map.get(term, []):
                if val > score_map.get(parent, 0.0):
                    score_map[parent] = val

        for term, score in score_map.items():
            if score >= SCORE_CUTOFF:
                results.append((protein_id, term, score))

    return pd.DataFrame(results, columns=["protein_id", "go_term", "score"])

# ==========================================
# PIPELINE
# ==========================================
topo_terms, parent_map = load_go_dag(GO_OBO_FILE)

print("[Step 2] Locating input TSV...")
SUBMISSION_PATH = find_submission_tsv()

submission_df = pd.read_csv(
    SUBMISSION_PATH,
    sep="\t",
    header=None,
    names=["protein_id", "go_term", "score"],
    dtype={"score": np.float32},
    on_bad_lines="skip"
)

final_df = enforce_hierarchy(submission_df, topo_terms, parent_map)

print(f"[Step 4] Saving output ({len(final_df)} rows)...")
final_df["score"] = final_df["score"].round(3)
final_df.to_csv(OUTPUT_TSV, sep="\t", index=False, header=False)

print("==========================================")
print("✔ DONE — submission.tsv GENERATED")
print("==========================================")
print(final_df.head())


[Step 1] Loading GO ontology...
[Step 2] Locating input TSV...
[Auto] Found TSV file: /kaggle/input/cafa6-goa-predictions/goa_submission.tsv
[Step 3] Propagating scores...


  0%|          | 0/279431 [00:00<?, ?it/s]

[Step 4] Saving output (54680279 rows)...
✔ DONE — submission.tsv GENERATED
   protein_id     go_term  score
0  A0A009IHW8  GO:0016787    1.0
1  A0A009IHW8  GO:0050135    1.0
2  A0A009IHW8  GO:0003824    1.0
3  A0A009IHW8  GO:0016798    1.0
4  A0A009IHW8  GO:0003674    1.0
